# Causal VAE Latent DAG Comprehensive Analysis

Comprehensive analysis of the Cross-Modal Causal LDVAE (Latent DAG) for multi-omics integration.

**Model Architecture:**
- **Encoder:** Deep residual encoders for RNA and Protein
- **SCM Layer:** Causal Structural Causal Model with Gumbel-Sigmoid structure learning
- **Decoder:** Linear LDVAE decoders for interpretable factor loadings
- **Modalities:** RNA-seq (2000 HVG genes) + Protein (228 proteins)
- **Latent Dimensions:** 128 RNA + 64 Protein

**Analysis Sections:**
1. Data Loading and Setup
2. Training Metrics
3. RNA Regulatory Network (GRN)
4. Protein Interaction Network (PPI)
5. Cross-Modal Interactions (Translation & Transcription)

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import normalized_mutual_info_score, silhouette_score, adjusted_rand_score, roc_auc_score, average_precision_score
from sklearn.cluster import KMeans
import umap
import networkx as nx
import warnings
import gseapy as gp
from scipy.stats import spearmanr, pearsonr
import scanpy as sc
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import community.community_louvain as community_louvain
from sklearn.metrics import silhouette_score
from scipy.stats import wasserstein_distance
from scipy.spatial.distance import cdist
from scipy.stats import mannwhitneyu
import decoupler as dc

# Import the model for inference
from VAE_latent_dag_v2 import CrossModalCausalLDVAE

warnings.filterwarnings('ignore')

# Plotting setup
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

sns.set_context("paper") 
sns.set_style("white")

# --- Global Style Variables for Consistency ---
TITLE_SIZE  = 22
LABEL_SIZE  = 18
TICK_SIZE   = 14
LEGEND_SIZE = 14
LINE_WIDTH  = 2.5

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Data Loading and Setup

In [ ]:
# Set model directory
MODEL_DIR = "latent_dag_vae_grid_search/results_combined_b4.0_rna1000_ortho5.0"

# Verify directory exists
if not os.path.exists(MODEL_DIR):
    raise FileNotFoundError(f"Model directory '{MODEL_DIR}' not found. Please update MODEL_DIR variable.")

print(f"Loading results from: {MODEL_DIR}")

In [ ]:
# Load raw data for reference
print("Loading raw data...")
rna_adata = sc.read_h5ad("data/combined/rna_combined.h5ad")
prot_adata = sc.read_h5ad("data/combined/citeseq_combined.h5ad")

# Align observations
common_obs = list(set(rna_adata.obs_names) & set(prot_adata.obs_names))
print(f"Aligning samples: {len(common_obs)} cells.")

rna_adata = rna_adata[common_obs].copy()
prot_adata = prot_adata[common_obs].copy()

gene_names = rna_adata.var_names.tolist()
protein_names = prot_adata.var_names.tolist()

print(f"Raw data loaded:")
print(f"  RNA-seq: {rna_adata.shape} ({len(gene_names)} genes)")
print(f"  Protein: {prot_adata.shape} ({len(protein_names)} proteins)")

# Cell type labels
if 'celltype.l1' in rna_adata.obs.columns:
    cell_labels = rna_adata.obs['celltype.l1'].values
    print(f"\nCell type distribution:")
    print(rna_adata.obs['celltype.l1'].value_counts())
else:
    cell_labels = None
    print("No cell type annotations found.")

In [ ]:
# Load priors from config.json (ensures they match the model's training priors)
print("\nLoading prior knowledge...")
with open(f"{MODEL_DIR}/config.json") as f:
    model_config = json.load(f)

rna_prior_path = model_config['prior_rna_path']
prot_prior_path = model_config['prior_prot_path']
print(f"  RNA prior:     {rna_prior_path}")
print(f"  Protein prior: {prot_prior_path}")

rna_priors = torch.load(rna_prior_path, map_location='cpu')
protein_priors = torch.load(prot_prior_path, map_location='cpu')

print(f"\nRNA prior shape: {rna_priors['prior_mask'].shape}")
print(f"Protein prior shape: {protein_priors['prior_mask'].shape}")

# Calculate prior sparsity
rna_prior_sparsity = (rna_priors['prior_mask'] == 0).sum().item() / rna_priors['prior_mask'].numel()
prot_prior_sparsity = (protein_priors['prior_mask'] == 0).sum().item() / protein_priors['prior_mask'].numel()

print(f"\nPrior Sparsity:")
print(f"  RNA: {rna_prior_sparsity:.4f}")
print(f"  Protein: {prot_prior_sparsity:.4f}")

# Store config flags for evaluation
prior_abs = model_config.get('prior_abs', False)
prior_norm = model_config.get('prior_norm', False)
print(f"\nTraining config: prior_abs={prior_abs}, prior_norm={prior_norm}")

## 2. Load Training Results and Inferred Networks

In [ ]:
# Load training metrics
with open(f"{MODEL_DIR}/metrics.json", 'r') as f:
    metrics = json.load(f)

print(f"Training history loaded:")
print(f"  Train epochs: {len(metrics['train'])}")
print(f"  Metric keys: {list(metrics['train'][0].keys())}")

In [ ]:
epochs = [m+1 for m in range(len(metrics['train']))]
train_m = {k: [m[k] for m in metrics['train']] for k in metrics['train'][0].keys() if k != 'epoch'}
val_m = {k: [m[k] for m in metrics['val']] for k in metrics['val'][0].keys() if k != 'epoch'}

fig, axes = plt.subplots(3, 3, figsize=(22, 18))
fig.suptitle('Latent Causal VAE Training Metrics', fontsize=TITLE_SIZE + 4, fontweight='bold', y=1.02)

plot_configs = [
    ('rec', 0, 0, 'Reconstruction Loss', 'Causal Reconstruction'),
    ('kl_r', 0, 1, 'KL Divergence', 'KL Divergence (RNA)'),
    ('kl_p', 0, 2, 'KL Divergence', 'KL Divergence (Protein)'),
    ('dag', 1, 0, 'DAGMA', 'DAGMA Acyclicity'),
    ('intra_sparse', 1, 1, 'Sparsity Loss', 'Intra-Modality Sparsity'),
    ('inter_sparse', 1, 2, 'Sparsity Loss', 'Inter-Modality Sparsity'),
    ('dec_sparse', 2, 0, 'Sparsity Loss', 'Decoder Sparsity'),
    ('ortho', 2, 1, 'Orthogonal Loss', 'Weight Orthogonality'),
    ('prior', 2, 2, 'Prior Loss', 'Biological Prior')
]

for key, r, c, ylabel, title in plot_configs:
    ax = axes[r, c]
    if key in train_m:
        ax.plot(epochs, train_m[key], label='Train', linewidth=LINE_WIDTH, color='#3498db')
        ax.plot(epochs, val_m[key], label='Val', linewidth=LINE_WIDTH, color='#e74c3c')
        
        ax.set_title(title, fontsize=TITLE_SIZE, fontweight='bold', pad=15)
        ax.set_xlabel('Epoch', fontsize=LABEL_SIZE)
        ax.set_ylabel(ylabel, fontsize=LABEL_SIZE)
        
        ax.tick_params(axis='both', which='major', labelsize=TICK_SIZE)
        ax.legend(fontsize=LEGEND_SIZE, frameon=False)
        ax.grid(False)

plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/training_metrics_comprehensive.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Load inferred networks
print("Loading inferred networks...")
grn = pd.read_csv(f"{MODEL_DIR}/GRN_inferred.csv", index_col=0)
ppi = pd.read_csv(f"{MODEL_DIR}/PPI_inferred.csv", index_col=0)
translation = pd.read_csv(f"{MODEL_DIR}/Transcription_RNA_to_Prot.csv", index_col=0)
transcription = pd.read_csv(f"{MODEL_DIR}/Regulation_Prot_to_RNA.csv", index_col=0)

print(f"\nInferred Networks:")
print(f"  GRN (Gene Regulatory Network): {grn.shape}")
print(f"  PPI (Protein-Protein Interaction): {ppi.shape}")
print(f"  Translation (Gene → Protein): {translation.shape}")
print(f"  Transcription (Protein → Gene): {transcription.shape}")

# Calculate sparsity
print(f"\nNetwork Statistics (Non-zero edges):")
print(f"  GRN: {(grn.values != 0).sum()} / {grn.size} ({(grn.values != 0).sum()/grn.size*100:.2f}%)")
print(f"  PPI: {(ppi.values != 0).sum()} / {ppi.size} ({(ppi.values != 0).sum()/ppi.size*100:.2f}%)")
print(f"  Translation: {(translation.values != 0).sum()} / {translation.size} ({(translation.values != 0).sum()/translation.size*100:.2f}%)")
print(f"  Transcription: {(transcription.values != 0).sum()} / {transcription.size} ({(transcription.values != 0).sum()/transcription.size*100:.2f}%)")

In [ ]:
def evaluate_prior_alignment(inferred_df, prior_mask, name="Network", use_abs=True):
    """Evaluate how well inferred network recovers biological prior edges.
    
    Args:
        use_abs: If True, use |edge weights|. If False, use signed values (matches training when prior_abs=False).
    """
    # --- Data Processing ---
    raw_flat = inferred_df.values.flatten()
    inferred_flat = np.abs(raw_flat) if use_abs else raw_flat
    
    if torch.is_tensor(prior_mask):
        prior_flat = prior_mask.detach().cpu().numpy().flatten()
    else:
        prior_flat = prior_mask.flatten()

    if inferred_flat.shape != prior_flat.shape:
        print(f"Shape mismatch for {name}: Inferred {inferred_flat.shape} vs Prior {prior_flat.shape}")
        return

    signal_indices = prior_flat == 1
    background_indices = prior_flat == 0

    signal_values = inferred_flat[signal_indices]
    background_values = inferred_flat[background_indices]

    # --- Metrics Calculations ---
    mode_label = "Absolute" if use_abs else "Signed"
    mean_signal = np.mean(signal_values)
    mean_background = np.mean(background_values)
    fold_enrichment = mean_signal / (mean_background + 1e-9) if use_abs else mean_signal / (np.abs(mean_background) + 1e-9)
    auroc = roc_auc_score(prior_flat, inferred_flat)
    auprc = average_precision_score(prior_flat, inferred_flat)
    
    # Random baseline AUPRC (= positive fraction)
    pos_frac = prior_flat.sum() / len(prior_flat)

    print(f"--- {name} [{mode_label}] ---")
    print(f"Mean Value (Prior=1): {mean_signal:.6f}")
    print(f"Mean Value (Prior=0): {mean_background:.6f}")
    print(f"Fold Enrichment: {fold_enrichment:.4f}")
    print(f"AUROC: {auroc:.4f}")
    print(f"AUPRC: {auprc:.4f}  (random baseline: {pos_frac:.6f}, enrichment: {auprc/pos_frac:.1f}x)")
    print("-" * 40)

    # --- Plotting ---
    plt.figure(figsize=(10, 7))
    
    plt.hist(background_values, bins=100, alpha=0.5, label='Prior=0 (Background)', 
             density=True, color='gray', log=True, linewidth=LINE_WIDTH)
    plt.hist(signal_values, bins=100, alpha=0.5, label='Prior=1 (Signal)', 
             density=True, color='red', log=True, linewidth=LINE_WIDTH)

    plt.title(f'{name}: Prior Recovery [{mode_label}]\nAUROC: {auroc:.3f} | Fold: {fold_enrichment:.2f}x', 
              fontsize=TITLE_SIZE, fontweight='bold', pad=20)
    xlabel = 'Absolute Edge Weight' if use_abs else 'Signed Edge Weight'
    plt.xlabel(xlabel, fontsize=LABEL_SIZE)
    plt.ylabel('Density (Log Scale)', fontsize=LABEL_SIZE)
    
    plt.xticks(fontsize=TICK_SIZE)
    plt.yticks(fontsize=TICK_SIZE)
    plt.legend(fontsize=LEGEND_SIZE, frameon=False)
    plt.grid(True, alpha=0.2, linestyle='--')
    
    plt.tight_layout()
    suffix = "abs" if use_abs else "signed"
    plt.savefig(f"{MODEL_DIR}/{name}_prior_recovery_{suffix}.png", dpi=300, bbox_inches='tight')
    plt.show()


# Evaluate both modes: signed (matches training) and absolute
for use_abs in [False, True]:
    evaluate_prior_alignment(grn, rna_priors['prior_mask'], name="GRN", use_abs=use_abs)
    evaluate_prior_alignment(ppi, protein_priors['prior_mask'], name="PPI", use_abs=use_abs)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

axes[0].hist(grn.values, bins=100, edgecolor='black', alpha=0.7, log=True)
axes[0].set_title('Flattened GRN Values')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Log Frequency')

axes[1].hist(ppi.values, bins=100, edgecolor='black', alpha=0.7, log=True)
axes[1].set_title('Flattened PPI Values')
axes[1].set_xlabel('Value')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

axes[0].hist(translation.values, bins=100, edgecolor='black', alpha=0.7, log=True)
axes[0].set_title('Flattened RNA --> Protein Values')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Log Frequency')

axes[1].hist(transcription.values, bins=100, edgecolor='black', alpha=0.7, log=True)
axes[1].set_title('Flattened Protein --> RNA Values')
axes[1].set_xlabel('Value')

plt.tight_layout()
plt.show()

In [ ]:
# Load factor loadings
print("\nLoading factor loadings...")
rna_factors = pd.read_csv(f"{MODEL_DIR}/RNA_Factor_Loadings.csv", index_col=0)
protein_factors = pd.read_csv(f"{MODEL_DIR}/Prot_Factor_Loadings.csv", index_col=0)
print(f"RNA Factor Loadings: {rna_factors.shape}")
print(f"  Genes (rows) × Latent Factors (columns)")

## 3. Extract Latent Representations

Load the trained model and extract latent representations for all cells.

In [ ]:
# Prepare data tensors
X_rna = rna_adata.X.toarray() if hasattr(rna_adata.X, 'toarray') else rna_adata.X
X_prot = prot_adata.X.toarray() if hasattr(prot_adata.X, 'toarray') else prot_adata.X

# Z-score normalization (as in training)
X_rna = (X_rna - X_rna.mean(axis=0)) / (X_rna.std(axis=0) + 1e-6)
X_prot = (X_prot - X_prot.mean(axis=0)) / (X_prot.std(axis=0) + 1e-6)

X_rna_tensor = torch.FloatTensor(X_rna).to(device)
X_prot_tensor = torch.FloatTensor(X_prot).to(device)

print(f"Data tensors prepared:")
print(f"  RNA: {X_rna_tensor.shape}")
print(f"  Protein: {X_prot_tensor.shape}")

In [ ]:
rna_adata

In [ ]:
rna_latent = pd.read_csv(f"{MODEL_DIR}/latents_causal_rna.csv", index_col = 0)
protein_latent = pd.read_csv(f"{MODEL_DIR}/latents_causal_prot.csv", index_col = 0)
rna_latent["celltype"] = rna_adata.obs["celltype"]
protein_latent["celltype"] = rna_adata.obs["celltype"]


rna_latent_epsilon = pd.read_csv(f"{MODEL_DIR}/latents_exogenous_rna.csv", index_col = 0)
protein_latent_epsilon = pd.read_csv(f"{MODEL_DIR}/latents_exogenous_prot.csv", index_col = 0)
rna_latent_epsilon["celltype"] = rna_adata.obs["celltype"]
protein_latent_epsilon["celltype"] = rna_adata.obs["celltype"]

In [ ]:
from matplotlib.patches import Patch
def render_weight_kde(df_input, title_prefix, ax, y_limit=None):
    """Helper function to handle the math and styling for a single axis."""
    long_df = df_input.reset_index().melt(id_vars='index')
    long_df.columns = ['gene', 'protein', 'weight']
    long_df['is_match'] = long_df['gene'] == long_df['protein']
    
    matches = long_df[long_df['is_match'] == True]['weight']
    background = long_df[long_df['is_match'] == False]['weight']
    stat, p_value = mannwhitneyu(matches, background, alternative='greater')

    # Set legend=False to prevent seaborn from drawing an incomplete default legend
    sns.kdeplot(
        data=long_df, x='weight', hue='is_match', 
        fill=True, common_norm=False, palette='Set2', 
        bw_adjust=0.5, linewidth=LINE_WIDTH, ax=ax,
        legend=False 
    )
    
    ax.set_title(f'{title_prefix}\n$p = {p_value:.2e}$', fontsize=TITLE_SIZE, fontweight='bold', pad=20)
    ax.set_xlabel('Edge Weight', fontsize=LABEL_SIZE)
    ax.set_ylabel('Density', fontsize=LABEL_SIZE)
    ax.tick_params(axis='both', which='major', labelsize=TICK_SIZE)
    
    if y_limit:
        ax.set_ylim(y_limit)
    
    # --- Custom Legend Implementation ---
    # In the 'Set2' palette, index 0 is green (False/Unmatched) and index 1 is orange (True/Matched)
    colors = sns.color_palette('Set2')
    
    legend_elements = [
        Patch(facecolor=colors[1], label='Matched gene-protein', alpha=0.6),
        Patch(facecolor=colors[0], label='Unmatched gene-protein', alpha=0.6)
    ]
    
    # Add the legend to the axis
    ax.legend(handles=legend_elements, fontsize=LEGEND_SIZE, frameon=False, loc='upper right')


def plot_and_save_all_distributions(df1, title1, df2, title2, base_save_path):
    # Split the path into the base name and extension (e.g., '.pdf' or '.png')
    base_name, ext = os.path.splitext(base_save_path)
    
    # 1. Create a temporary figure just to calculate the shared y-axis limits
    fig_temp, axes = plt.subplots(2, 1)
    render_weight_kde(df1, title1, axes[0])
    render_weight_kde(df2, title2, axes[1])
    
    # Get the max y-limit across both plots so they are on the same scale
    y_lim_0 = axes[0].get_ylim()
    y_lim_1 = axes[1].get_ylim()
    common_y_lim = (min(y_lim_0[0], y_lim_1[0]), max(y_lim_0[1], y_lim_1[1]))
    
    # Close the temporary combined figure without saving it
    plt.close(fig_temp) 

    # 2. Generate and save the individual plots
    datasets = [(df1, title1), (df2, title2)]
    for i, (df, title) in enumerate(datasets, 1):
        fig, ax = plt.subplots(figsize=(8, 6))
        
        # Apply the common y-limit
        render_weight_kde(df, title, ax, y_limit=common_y_lim)
        
        # Create the new filename: e.g., "my_plot_1.pdf"
        individual_path = f"{base_name}_{i}{ext}"
        
        plt.tight_layout()
        plt.savefig(individual_path, dpi=300, bbox_inches='tight')
        plt.show()  # Display in the notebook
        plt.close(fig)
        
        print(f"Saved individual plot {i} to: {individual_path}")

# Execute
plot_and_save_all_distributions(
    translation, "RNA $\\rightarrow$ Protein", 
    transcription, "Protein $\\rightarrow$ RNA", 
    f"{MODEL_DIR}/rna_protein_weight_distribution.pdf"
)

In [ ]:
collectri = dc.op.collectri(organism='human')

valid_genes = set(grn.index)
valid_collectri = collectri[
    collectri['source'].isin(valid_genes) & 
    collectri['target'].isin(valid_genes)
].drop_duplicates(subset=['source', 'target'])

n_pairs = len(valid_collectri)
# ---> MODIFICATION: Calculate unique TFs <---
n_unique_tfs = valid_collectri['source'].nunique()

print(f"Valid pairs: {n_pairs}")
print(f"Unique TFs: {n_unique_tfs}")

# Extract Weights
tf_indices = [grn.index.get_loc(tf) for tf in valid_collectri['source']]
gene_indices = [grn.index.get_loc(gene) for gene in valid_collectri['target']]

forward_weights = np.abs(grn.values[tf_indices, gene_indices])
reverse_weights = np.abs(grn.values[gene_indices, tf_indices])

# --- STATISTICS ---
stat, p_val = mannwhitneyu(forward_weights, reverse_weights, alternative='greater')
print(f"P-value: {p_val:.2e}")

# Prepare DataFrame for Boxplot
df_plot = pd.DataFrame({
    'Weight': np.concatenate([forward_weights, reverse_weights]),
    'Direction': [f'TF $\\to$ Gene\n(Ground Truth)\n$n={n_pairs}$'] * len(forward_weights) + 
                 [f'Gene $\\to$ TF\n(Reverse)\n$n={n_pairs}$'] * len(reverse_weights)
})

# --- PLOTTING (BOXPLOT) ---
plt.figure(figsize=(8, 7))

# Boxplot with log scale
ax = sns.boxplot(x='Direction', y='Weight', data=df_plot, 
                 palette=['forestgreen', 'firebrick'], 
                 width=0.5, fliersize=1, linewidth=LINE_WIDTH)

plt.yscale('log')

# Formatting
plt.title("Causal Edge Strength: TF vs Gene", fontsize=TITLE_SIZE)
plt.ylabel("Absolute Edge Weight (Log Scale)", fontsize=LABEL_SIZE)
plt.xlabel("", fontsize=LABEL_SIZE)
plt.xticks(fontsize=TICK_SIZE)
plt.yticks(fontsize=TICK_SIZE)

# ---> MODIFICATION: Add text box for Unique TFs in the upper left corner <---
# transform=ax.transAxes sets the coordinates relative to the plot axes (0,0 is bottom left, 1,1 is top right)
ax.text(0.05, 0.95, f'{n_unique_tfs} Unique TFs', 
        transform=ax.transAxes, fontsize=LEGEND_SIZE, 
        verticalalignment='top', 
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray', alpha=0.8))

# Add P-value bracket
x1, x2 = 0, 1
y, h, col = df_plot['Weight'].max() * 1.5, 0, 'k'

plt.ylim(top=y * 10)

# Draw bracket
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)

plt.text((x1+x2)*.5, y*1.1, f"Mann-Whitney\n$P < {p_val:.2e}$", 
         ha='center', va='bottom', color=col, fontsize=LEGEND_SIZE)

plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/collectri_causal_boxplot.pdf", dpi=300)
plt.savefig(f"{MODEL_DIR}/collectri_causal_boxplot.png", dpi=300)
plt.show()

In [ ]:
def process_umap(df):
    reducer = umap.UMAP(random_state=42)
    embedding = reducer.fit_transform(df.drop(columns=['celltype']))
    return pd.DataFrame(embedding, columns=['U1', 'U2'], index=df.index).assign(celltype=df['celltype'])

u_rna = process_umap(rna_latent)
u_prot = process_umap(protein_latent)
u_rna_eps = process_umap(rna_latent_epsilon)
u_prot_eps = process_umap(protein_latent_epsilon)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
configs = [
    (u_rna, "RNA Latent", axes[0, 0]),
    (u_prot, "Protein Latent", axes[0, 1]),
    (u_rna_eps, "RNA Exogenous Error", axes[1, 0]),
    (u_prot_eps, "Protein Exogenous Error", axes[1, 1])
]

for data, title, ax in configs:
    sns.scatterplot(data=data, x='U1', y='U2', hue='celltype', ax=ax, s=8, palette='tab20', edgecolor=None)
    
    ax.set_title(title, fontsize=TITLE_SIZE, fontweight='bold', pad=15)
    ax.set_xlabel("UMAP 1", fontsize=LABEL_SIZE)
    ax.set_ylabel("UMAP 2", fontsize=LABEL_SIZE)
    
    ax.tick_params(axis='both', which='major', labelsize=TICK_SIZE)
    
    ax.legend(
        bbox_to_anchor=(1.05, 1), 
        loc='upper left', 
        fontsize=LEGEND_SIZE, 
        markerscale=3,
        frameon=False
    )

plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/latent_umaps.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def evaluate_disentanglement(z_df, eps_df, label_col='celltype', sample_size=5000):
    """
    Compares Latent Z vs Exogenous Epsilon on Clustering and Gaussianity.
    """
    results = {}
    
    # Subsample for Silhouette Score (it's slow on 160k cells)
    if len(z_df) > sample_size:
        idx = np.random.choice(len(z_df), sample_size, replace=False)
        z_sub = z_df.iloc[idx]
        eps_sub = eps_df.iloc[idx]
    else:
        z_sub = z_df
        eps_sub = eps_df
        
    labels = z_sub[label_col]
    z_data = z_sub.drop(columns=[label_col]).values
    eps_data = eps_sub.drop(columns=[label_col]).values
    
    print(f"--- Computing Metrics (n={len(z_sub)}) ---")
    
    # 1. CLUSTER TIGHTNESS (Silhouette Score)
    # Range: -1 to 1. Higher is better defined clusters.
    print("Calculating Silhouette Scores...")
    sil_z = silhouette_score(z_data, labels)
    sil_eps = silhouette_score(eps_data, labels)
    
    # 2. GAUSSIANITY (Wasserstein Distance to N(0,1))
    # We compare the flattened distribution of values to a theoretical Normal distribution
    # Lower = Closer to Gaussian Noise
    print("Calculating Gaussianity...")
    
    # Flatten all latent dimensions to get the global distribution of values
    z_flat = z_data.flatten()
    eps_flat = eps_data.flatten()
    
    # Generate theoretical normal distribution for comparison
    ref_normal = np.random.randn(len(z_flat))
    
    wass_z = wasserstein_distance(z_flat, ref_normal)
    wass_eps = wasserstein_distance(eps_flat, ref_normal)
    
    results = {
        "Metric": ["Silhouette Score (Clustering)", "Wasserstein Dist to N(0,1)"],
        "Latent Z": [sil_z, wass_z],
        "Exogenous Eps": [sil_eps, wass_eps],
        "Delta (Z - Eps)": [sil_z - sil_eps, wass_z - wass_eps],
        "Interpretation": [
            "Positive = Z is more clustered", 
            "Positive = Eps is more Gaussian"
        ]
    }
    
    return pd.DataFrame(results)

In [ ]:
print("\n=== RNA MODALITY EVALUATION ===")
df_rna_eval = evaluate_disentanglement(rna_latent, rna_latent_epsilon, sample_size = len(rna_latent))
print(df_rna_eval.to_string(index=False))

print("\n\n=== PROTEIN MODALITY EVALUATION ===")
df_prot_eval = evaluate_disentanglement(protein_latent, protein_latent_epsilon, sample_size = len(protein_latent))
print(df_prot_eval.to_string(index=False))

## 5. RNA Regulatory Network (GRN) Analysis

In [ ]:
def suggest_threshold(weights_df, modality_name):
    """Suggest thresholding values for network sparsification"""
    w_flat = weights_df.values.flatten()
    w_abs = np.abs(w_flat)
    
    w_nonzero = w_abs[w_abs > 1e-6]
    sigma = np.std(w_nonzero)
    thresh_3sigma = 3 * sigma
    
    print(f"--- {modality_name} Thresholds ---")
    print(f"Standard Deviation (sigma): {sigma:.4f}")
    print(f"Recommended 3-Sigma Threshold: {thresh_3sigma:.4f}")
    
    thresh_99 = np.percentile(w_abs, 99.0)
    thresh_95 = np.percentile(w_abs, 95.0)
    print(f"Top 1% Threshold: {thresh_99:.4f}")
    print(f"Top 5% Threshold: {thresh_95:.4f}")
    
    return thresh_3sigma

THRESHOLD_GRN = suggest_threshold(grn, "GRN")
THRESHOLD_PPI = suggest_threshold(ppi, "PPI")
THRESHOLD_TRANSLATION = suggest_threshold(translation, "Translation")
THRESHOLD_TRANSCRIPTION = suggest_threshold(transcription, "Transcription")

In [ ]:
# Apply thresholds
grn_thresh = grn.copy()
ppi_thresh = ppi.copy()
translation_thresh = translation.copy()
transcription_thresh = transcription.copy()

grn_thresh[np.abs(grn_thresh) < THRESHOLD_GRN] = 0
ppi_thresh[np.abs(ppi_thresh) < THRESHOLD_PPI] = 0
translation_thresh[np.abs(translation_thresh) < THRESHOLD_TRANSLATION] = 0
transcription_thresh[np.abs(transcription_thresh) < THRESHOLD_TRANSCRIPTION] = 0

print("Network Statistics After Thresholding:")
print(f"\nGRN: {(grn_thresh.values != 0).sum()} edges ({1 - (grn_thresh.values != 0).sum() / grn_thresh.size:.4f} sparsity)")
print(f"PPI: {(ppi_thresh.values != 0).sum()} edges ({1 - (ppi_thresh.values != 0).sum() / ppi_thresh.size:.4f} sparsity)")
print(f"Translation: {(translation_thresh.values != 0).sum()} edges ({1 - (translation_thresh.values != 0).sum() / translation_thresh.size:.4f} sparsity)")
print(f"Transcription: {(transcription_thresh.values != 0).sum()} edges ({1 - (transcription_thresh.values != 0).sum() / transcription_thresh.size:.4f} sparsity)")

In [ ]:
# Compute gene regulatory statistics
grn_mat = grn_thresh.values

# Out-degree: number of genes each gene regulates
out_degree = np.sum(grn_mat != 0, axis=1)
out_strength = np.sum(np.abs(grn_mat), axis=1)

# In-degree: number of genes regulating each gene
in_degree = np.sum(grn_mat != 0, axis=0)
in_strength = np.sum(np.abs(grn_mat), axis=0)

gene_stats = pd.DataFrame({
    'Gene': gene_names,
    'Out_Degree': out_degree,
    'Out_Strength': out_strength,
    'In_Degree': in_degree,
    'In_Strength': in_strength
})

gene_stats['Total_Degree'] = gene_stats['Out_Degree'] + gene_stats['In_Degree']
gene_stats = gene_stats.sort_values('Total_Degree', ascending=False)

print("=== Top 20 Hub Genes (by Total Degree) ===")
print(gene_stats.head(20).to_string(index=False))

# Save gene statistics
gene_stats.to_csv(f"{MODEL_DIR}/gene_network_statistics.csv", index=False)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('Gene Regulatory Network (GRN) Analysis', fontsize=TITLE_SIZE + 4, fontweight='bold', y=1.02)

axes[0, 0].hist(out_degree[out_degree > 0], bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Out-Degree', fontsize=LABEL_SIZE)
axes[0, 0].set_ylabel('Frequency', fontsize=LABEL_SIZE)
axes[0, 0].set_title('Out-Degree Distribution (Regulators)', fontsize=TITLE_SIZE, fontweight='bold', pad=15)
axes[0, 0].tick_params(axis='both', labelsize=TICK_SIZE)
axes[0, 0].grid(False)

axes[0, 1].hist(in_degree[in_degree > 0], bins=50, color='#e74c3c', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('In-Degree', fontsize=LABEL_SIZE)
axes[0, 1].set_ylabel('Frequency', fontsize=LABEL_SIZE)
axes[0, 1].set_title('In-Degree Distribution (Targets)', fontsize=TITLE_SIZE, fontweight='bold', pad=15)
axes[0, 1].tick_params(axis='both', labelsize=TICK_SIZE)
axes[0, 1].grid(False)

scatter = axes[1, 0].scatter(gene_stats['In_Strength'], gene_stats['Out_Strength'],
                            s=80, alpha=0.6, c=gene_stats['Total_Degree'], 
                            cmap='viridis', edgecolors='black', linewidth=0.5)
axes[1, 0].set_xlabel('In-Strength (Regulated)', fontsize=LABEL_SIZE)
axes[1, 0].set_ylabel('Out-Strength (Regulating)', fontsize=LABEL_SIZE)
axes[1, 0].set_title('Gene Hub Analysis', fontsize=TITLE_SIZE, fontweight='bold', pad=15)
axes[1, 0].tick_params(axis='both', labelsize=TICK_SIZE)
axes[1, 0].grid(False)

cbar = plt.colorbar(scatter, ax=axes[1, 0])
cbar.set_label('Total Degree', fontsize=LABEL_SIZE)
cbar.ax.tick_params(labelsize=TICK_SIZE)

top_hubs = gene_stats.head(10)
for idx, row in top_hubs.iterrows():
    axes[1, 0].annotate(row['Gene'][:10], 
                       (row['In_Strength'], row['Out_Strength']),
                       textcoords="offset points", xytext=(0,7), 
                       ha='center', fontsize=TICK_SIZE - 2, fontweight='bold')

top_regulators = gene_stats.nlargest(20, 'Out_Strength')
axes[1, 1].barh(range(len(top_regulators)), top_regulators['Out_Strength'].values, 
                color='#2ecc71', edgecolor='black', alpha=0.7)
axes[1, 1].set_yticks(range(len(top_regulators)))
axes[1, 1].set_yticklabels(top_regulators['Gene'].values, fontsize=TICK_SIZE)
axes[1, 1].set_xlabel('Out-Strength', fontsize=LABEL_SIZE)
axes[1, 1].set_title('Top 20 Master Regulators', fontsize=TITLE_SIZE, fontweight='bold', pad=15)
axes[1, 1].tick_params(axis='x', labelsize=TICK_SIZE)
axes[1, 1].grid(False)
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/grn_network_analysis.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
top_50_genes = gene_stats.head(50)['Gene'].tolist()
grn_subset = grn_thresh.loc[top_50_genes, top_50_genes]

plt.figure(figsize=(16, 14))
sns.heatmap(grn_subset, cmap='RdBu_r', center=0, 
            xticklabels=True, yticklabels=True,
            cbar_kws={'label': 'Regulatory Weight'})

ax = plt.gca()
ax.figure.axes[-1].yaxis.label.set_size(LABEL_SIZE)
ax.figure.axes[-1].tick_params(labelsize=TICK_SIZE)

plt.title('GRN Subnetwork (Top 50 Hub Genes)', fontsize=TITLE_SIZE, fontweight='bold', pad=25)
plt.xlabel('Target Gene', fontsize=LABEL_SIZE, labelpad=15)
plt.ylabel('Source Gene', fontsize=LABEL_SIZE, labelpad=15)

plt.xticks(rotation=90, fontsize=TICK_SIZE - 4)
plt.yticks(rotation=0, fontsize=TICK_SIZE - 4)

plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/grn_heatmap_top50.png", dpi=300, bbox_inches='tight')
plt.show()

## 6. Protein Interaction Network (PPI) Analysis

In [ ]:
# Build protein network graph
protein_mat = ppi_thresh.values
n_proteins = protein_mat.shape[0]

G_protein = nx.DiGraph()
for i in range(n_proteins):
    G_protein.add_node(i, name=protein_names[i])
    for j in range(n_proteins):
        if protein_mat[i, j] != 0:
            G_protein.add_edge(i, j, weight=protein_mat[i, j])

print(f"Protein Network:")
print(f"  Nodes: {G_protein.number_of_nodes()}")
print(f"  Edges: {G_protein.number_of_edges()}")
print(f"  Is DAG: {nx.is_directed_acyclic_graph(G_protein)}")

In [ ]:
# Compute protein centrality measures
protein_in_degree = dict(G_protein.in_degree())
protein_out_degree = dict(G_protein.out_degree())

protein_in_strength = {i: sum(abs(G_protein[j][i]['weight']) 
                              for j in G_protein.predecessors(i)) 
                      for i in G_protein.nodes()}
protein_out_strength = {i: sum(abs(G_protein[i][j]['weight']) 
                               for j in G_protein.successors(i)) 
                       for i in G_protein.nodes()}

protein_stats = pd.DataFrame({
    'Protein': protein_names,
    'In_Degree': [protein_in_degree.get(i, 0) for i in range(n_proteins)],
    'Out_Degree': [protein_out_degree.get(i, 0) for i in range(n_proteins)],
    'In_Strength': [protein_in_strength.get(i, 0) for i in range(n_proteins)],
    'Out_Strength': [protein_out_strength.get(i, 0) for i in range(n_proteins)]
})

protein_stats['Total_Degree'] = protein_stats['In_Degree'] + protein_stats['Out_Degree']
protein_stats = protein_stats.sort_values('Total_Degree', ascending=False)

print("=== Top 20 Hub Proteins ===")
print(protein_stats.head(20).to_string(index=False))

# Save protein statistics
protein_stats.to_csv(f"{MODEL_DIR}/protein_network_statistics.csv", index=False)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('Protein Interaction Network (PPI) Analysis', fontsize=TITLE_SIZE + 4, fontweight='bold', y=1.02)

nonzero_weights = protein_mat[protein_mat != 0]
axes[0, 0].hist(nonzero_weights, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_title(f'Edge Weight Distribution (n={len(nonzero_weights)})', fontsize=TITLE_SIZE, fontweight='bold', pad=15)
axes[0, 0].set_xlabel('Weight', fontsize=LABEL_SIZE)
axes[0, 0].set_ylabel('Frequency', fontsize=LABEL_SIZE)
axes[0, 0].tick_params(axis='both', labelsize=TICK_SIZE)
axes[0, 0].grid(False)

axes[0, 1].hist(protein_stats['In_Degree'], bins=30, color='#e74c3c', alpha=0.5, label='In-Degree', edgecolor='black')
axes[0, 1].hist(protein_stats['Out_Degree'], bins=30, color='#2ecc71', alpha=0.5, label='Out-Degree', edgecolor='black')
axes[0, 1].set_title('Degree Distributions', fontsize=TITLE_SIZE, fontweight='bold', pad=15)
axes[0, 1].set_xlabel('Degree', fontsize=LABEL_SIZE)
axes[0, 1].set_ylabel('Frequency', fontsize=LABEL_SIZE)
axes[0, 1].tick_params(axis='both', labelsize=TICK_SIZE)
axes[0, 1].legend(fontsize=LEGEND_SIZE, frameon=False)
axes[0, 1].grid(False)

scatter = axes[1, 0].scatter(protein_stats['In_Strength'], 
                             protein_stats['Out_Strength'],
                             s=150, alpha=0.6, 
                             c=protein_stats['Total_Degree'], 
                             cmap='viridis', edgecolors='black')
axes[1, 0].set_xlabel('In-Strength', fontsize=LABEL_SIZE)
axes[1, 0].set_ylabel('Out-Strength', fontsize=LABEL_SIZE)
axes[1, 0].set_title('Protein Hub Analysis', fontsize=TITLE_SIZE, fontweight='bold', pad=15)
axes[1, 0].tick_params(axis='both', labelsize=TICK_SIZE)
axes[1, 0].grid(False)

cbar = plt.colorbar(scatter, ax=axes[1, 0])
cbar.set_label('Total Degree', fontsize=LABEL_SIZE)
cbar.ax.tick_params(labelsize=TICK_SIZE)

top_protein_hubs = protein_stats.head(5)
for idx, row in top_protein_hubs.iterrows():
    axes[1, 0].annotate(row['Protein'][:15], 
                       (row['In_Strength'], row['Out_Strength']),
                       textcoords="offset points", xytext=(0,12), 
                       ha='center', fontsize=TICK_SIZE, fontweight='bold')

top_20_proteins = protein_stats.head(20)
axes[1, 1].barh(range(len(top_20_proteins)), top_20_proteins['Total_Degree'].values,
               color='#9b59b6', edgecolor='black', alpha=0.7)
axes[1, 1].set_yticks(range(len(top_20_proteins)))
axes[1, 1].set_yticklabels(top_20_proteins['Protein'].values, fontsize=TICK_SIZE)
axes[1, 1].set_xlabel('Total Degree', fontsize=LABEL_SIZE)
axes[1, 1].set_title('Top 20 Hub Proteins', fontsize=TITLE_SIZE, fontweight='bold', pad=15)
axes[1, 1].tick_params(axis='x', labelsize=TICK_SIZE)
axes[1, 1].grid(False)
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/ppi_network_analysis.png", dpi=300, bbox_inches='tight')
plt.show()

## 7. Cross-Modal Interactions (Translation & Transcription)

In [ ]:
# Translation statistics (Gene → Protein)
translation_mat = translation_thresh.values
print("Translation (Gene → Protein):")
print(f"  Non-zero edges: {(translation_mat != 0).sum()}")
print(f"  Sparsity: {1 - (translation_mat != 0).sum() / translation_mat.size:.4f}")

# Top translating genes
translation_strength = np.sum(np.abs(translation_mat), axis=1)
top_translating = pd.DataFrame({
    'Gene': gene_names,
    'Translation_Strength': translation_strength
}).sort_values('Translation_Strength', ascending=False)

print("\nTop 10 Genes with Strongest Translation Effects:")
print(top_translating.head(10).to_string(index=False))

# Transcription statistics (Protein → Gene)
transcription_mat = transcription_thresh.values
print("\nTranscription (Protein → Gene):")
print(f"  Non-zero edges: {(transcription_mat != 0).sum()}")
print(f"  Sparsity: {1 - (transcription_mat != 0).sum() / transcription_mat.size:.4f}")

# Top transcribing proteins
transcription_strength = np.sum(np.abs(transcription_mat), axis=1)
top_transcribing = pd.DataFrame({
    'Protein': protein_names,
    'Transcription_Strength': transcription_strength
}).sort_values('Transcription_Strength', ascending=False)

print("\nTop 10 Proteins with Strongest Transcription Effects:")
print(top_transcribing.head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('Cross-Modal Interactions', fontsize=TITLE_SIZE + 4, fontweight='bold', y=1.02)

top_genes_translation = top_translating.head(50)['Gene'].tolist()
top_prots_translation = top_transcribing.head(50)['Protein'].tolist()
translation_subset = translation_thresh.loc[top_genes_translation, top_prots_translation]

vmax = max(np.abs(translation_thresh.values).max(), np.abs(transcription_thresh.values).max())

sns.heatmap(translation_subset, cmap='RdBu_r', center=0, ax=axes[0, 0],
            xticklabels=False, yticklabels=False, vmin=-vmax, vmax=vmax,
            cbar_kws={'label': 'Weight'})

cbar0 = axes[0, 0].collections[0].colorbar
cbar0.set_label('Weight', fontsize=LABEL_SIZE)
cbar0.ax.tick_params(labelsize=TICK_SIZE)

axes[0, 0].set_title('Translation Matrix (Gene → Protein)\nTop 50 × Top 50', fontsize=TITLE_SIZE, fontweight='bold', pad=15)
axes[0, 0].set_xlabel('Target Protein', fontsize=LABEL_SIZE)
axes[0, 0].set_ylabel('Source Gene', fontsize=LABEL_SIZE)

transcription_subset = transcription_thresh.loc[top_prots_translation, top_genes_translation]

sns.heatmap(transcription_subset, cmap='RdBu_r', center=0, ax=axes[0, 1],
            xticklabels=False, yticklabels=False, vmin=-vmax, vmax=vmax,
            cbar_kws={'label': 'Weight'})

cbar1 = axes[0, 1].collections[0].colorbar
cbar1.set_label('Weight', fontsize=LABEL_SIZE)
cbar1.ax.tick_params(labelsize=TICK_SIZE)

axes[0, 1].set_title('Transcription Matrix (Protein → Gene)\nTop 50 × Top 50', fontsize=TITLE_SIZE, fontweight='bold', pad=15)
axes[0, 1].set_xlabel('Target Gene', fontsize=LABEL_SIZE)
axes[0, 1].set_ylabel('Source Protein', fontsize=LABEL_SIZE)

axes[1, 0].hist(translation_strength[translation_strength > 0], bins=50, 
               color='coral', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Translation Strength', fontsize=LABEL_SIZE)
axes[1, 0].set_ylabel('Frequency', fontsize=LABEL_SIZE)
axes[1, 0].set_title('Gene Translation Strength Distribution', fontsize=TITLE_SIZE, fontweight='bold', pad=15)
axes[1, 0].tick_params(axis='both', labelsize=TICK_SIZE)
axes[1, 0].grid(False)

axes[1, 1].hist(transcription_strength[transcription_strength > 0], bins=50, 
               color='mediumseagreen', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Transcription Strength', fontsize=LABEL_SIZE)
axes[1, 1].set_ylabel('Frequency', fontsize=LABEL_SIZE)
axes[1, 1].set_title('Protein Transcription Strength Distribution', fontsize=TITLE_SIZE, fontweight='bold', pad=15)
axes[1, 1].tick_params(axis='both', labelsize=TICK_SIZE)
axes[1, 1].grid(False)

plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/cross_modal_interactions.png", dpi=300, bbox_inches='tight')
plt.show()

## Causality Validation Pathways

In [ ]:
causal_pathway_prot_prot = pd.read_csv(f"{MODEL_DIR}/causal_validation_pathways/csvs/top_causal_pathways_PROT_to_PROT.csv")
causal_pathway_prot_prot

In [ ]:
causal_pathway_rna_rna = pd.read_csv(f"{MODEL_DIR}/causal_validation_pathways/csvs/top_causal_pathways_RNA_to_RNA.csv")
causal_pathway_rna_rna

In [ ]:
causal_pathway_rna_prot = pd.read_csv(f"{MODEL_DIR}/causal_validation_pathways/csvs/top_causal_pathways_RNA_to_PROT.csv")
causal_pathway_rna_prot

In [ ]:
causal_pathway_prot_rna = pd.read_csv(f"{MODEL_DIR}/causal_validation_pathways/csvs/top_causal_pathways_PROT_to_RNA.csv")
causal_pathway_prot_rna

## Top 20 Edges Pathways

In [ ]:
grn_top20 = pd.read_csv(f"{MODEL_DIR}/biological_pathways/GRN_top20_edges.csv")
grn_top20

In [ ]:
ppi_top20 = pd.read_csv(f"{MODEL_DIR}/biological_pathways/PPI_top20_edges.csv")
ppi_top20

In [ ]:
regulation_top20 = pd.read_csv(f"{MODEL_DIR}/biological_pathways/Regulation_top20_edges.csv")
regulation_top20

In [ ]:
translation_top20 = pd.read_csv(f"{MODEL_DIR}/biological_pathways/Translation_top20_edges.csv")
translation_top20